# Demo — Onde o vetor mora: Azure SQL Database

**Objetivo:** guardar e buscar os vetores do acervo no **Azure SQL Database**, com o tipo nativo `VECTOR` e a função `VECTOR_DISTANCE`, e ver o que muda (e o que não muda) quando o vetor mora no banco relacional que a empresa já opera. Ilustra os slides 20 a 23: a pergunta que decide onde o vetor mora (20 e 21), a linha do Azure SQL na tabela das opções (22) e o DiskANN no triângulo dos algoritmos (23).

1. **Preparar os vetores** → os mesmos trechos e o mesmo embedding das outras bases da série;
2. **Criar a estrutura vetorial** → uma tabela comum, com uma coluna `VECTOR(n)`;
3. **Gravar os trechos** → `MERGE` pelo `id` do trecho: rodar de novo não duplica;
4. **Buscar por similaridade** → busca exata com `VECTOR_DISTANCE`, e o quase empate do SKU continua;
5. **RAG completo** → a resposta com citação, recuperada do banco da casa.

> **Cenário 100% fictício.** A Contoso Industrial e todos os documentos, projetos, pessoas e números do `acervo/` existem apenas para fins didáticos.

É um dos seis notebooks de `bases-vetoriais`, todos com o mesmo roteiro: compare-os lado a lado.

**Pré-requisito:** um banco no **Azure SQL Database** (o free tier, um General Purpose serverless, basta para a aula), com regra de firewall liberando o IP desta máquina, e o **ODBC Driver 17 ou 18 for SQL Server** instalado. O tipo `VECTOR` é GA no Azure SQL Database e não exige feature flag.

## 0. Preparação

Variáveis necessárias:

- `AZURE_OPENAI_ENDPOINT` e `AZURE_OPENAI_API_KEY`
- `AZURE_OPENAI_DEPLOYMENT` (deployment de chat, usado na Execução 5)
- `AZURE_OPENAI_EMBEDDING_DEPLOYMENT`, `AZURE_OPENAI_EMBEDDING_MODEL` e `AZURE_OPENAI_EMBEDDING_DIMENSIONS` (aqui, no máximo 1.998; o `.env.example` usa 1.536)
- `AZURE_SQL_CONNECTION_STRING`: a string ODBC do banco, com o nome do driver dentro dela

Os valores vêm do arquivo `Cap05/.env` (copie o `.env.example` e preencha); o que faltar é perguntado na hora da execução.

Sobre a string de conexão:

- **Driver:** `ODBC Driver 17 for SQL Server` e `ODBC Driver 18 for SQL Server` servem; use o que estiver instalado (o `.env.example` vem com o 18). A célula de conexão confere e, se faltar, lista os drivers desta máquina.
- **Criptografia:** mantenha `Encrypt=yes;TrustServerCertificate=no`. No Driver 17 o padrão de `Encrypt` é `no`, por isso ele precisa estar explícito; no 18 o padrão passou a ser `yes`. `TrustServerCertificate=no` mantém a validação do certificado do servidor.
- **Credenciais:** usuário e senha na string (autenticação SQL) simplificam a aula. Em produção, prefira Microsoft Entra ID, com a keyword `Authentication` do driver (por exemplo, `ActiveDirectoryMsi` para identidade gerenciada).

**Banco pausado:** um banco serverless, como o do free tier, pausa sozinho depois de um período sem conexão (no General Purpose, 60 minutos por padrão). Nesse caso, a primeira conexão pode devolver o erro 40613 enquanto ele retoma, o que em geral leva menos de um minuto: rode a célula de conexão de novo.

> Não grave chaves diretamente no notebook.

In [ ]:
%pip install -q "openai>=2" "python-dotenv>=1.0" "pandas>=2.1" "pyodbc>=5.1"

In [ ]:
import os
from getpass import getpass
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

load_dotenv(find_dotenv(usecwd=True))  # lê o .env da pasta Cap05, se existir

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT") or input(
    "AZURE_OPENAI_ENDPOINT (ex.: https://meu-recurso.openai.azure.com): "
).strip()
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY") or getpass("AZURE_OPENAI_API_KEY: ")
AZURE_OPENAI_DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT") or input(
    "AZURE_OPENAI_DEPLOYMENT (deployment de chat): "
).strip()
AZURE_OPENAI_EMBEDDING_DEPLOYMENT = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT") or input(
    "AZURE_OPENAI_EMBEDDING_DEPLOYMENT (deployment de embedding): "
).strip()

# O modelo e a dimensão do embedding são o contrato entre indexação e consulta.
EMBEDDING_MODEL = os.getenv("AZURE_OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")
EMBEDDING_DIMENSIONS = int(os.getenv("AZURE_OPENAI_EMBEDDING_DIMENSIONS", "1536"))

client = OpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    base_url=f"{AZURE_OPENAI_ENDPOINT.rstrip('/')}/openai/v1/",
)

# Pasta Cap05: a primeira pasta, subindo a partir daqui, que contém o acervo.
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "acervo").is_dir())
PASTA_ACERVO = RAIZ / "acervo"

print("Cliente configurado.")
print("Deployment de chat:", AZURE_OPENAI_DEPLOYMENT)
print("Embedding:", AZURE_OPENAI_EMBEDDING_DEPLOYMENT, f"({EMBEDDING_MODEL}, {EMBEDDING_DIMENSIONS} dimensões)")
print("Acervo:", PASTA_ACERVO)

In [ ]:
import re

import pyodbc

AZURE_SQL_CONNECTION_STRING = os.getenv("AZURE_SQL_CONNECTION_STRING") or getpass(
    "AZURE_SQL_CONNECTION_STRING (string ODBC, com usuário e senha): "
)

# O nome do driver ODBC vem dentro da string, e ele precisa estar instalado nesta máquina.
driver = re.search(r"Driver=\{([^}]*)\}", AZURE_SQL_CONNECTION_STRING, flags=re.IGNORECASE)
if driver and driver.group(1) not in pyodbc.drivers():
    raise RuntimeError(
        f"O driver '{driver.group(1)}' não está instalado. Drivers ODBC desta máquina: {pyodbc.drivers()}. "
        "Ajuste o Driver= da AZURE_SQL_CONNECTION_STRING: o 17 e o 18 servem."
    )

TABELA = "dbo.cap05_trechos"
# Banco serverless pausado: a conexão pode falhar com o erro 40613 enquanto ele retoma; rode a célula de novo.
conexao = pyodbc.connect(AZURE_SQL_CONNECTION_STRING)
banco, versao = conexao.cursor().execute("SELECT DB_NAME(), @@VERSION").fetchone()

print("Conectado ao banco:", banco)
print("Servidor:", versao.splitlines()[0].strip())
print("Driver ODBC em uso:", conexao.getinfo(pyodbc.SQL_DRIVER_NAME), conexao.getinfo(pyodbc.SQL_DRIVER_VER))
print("Tabela deste notebook:", TABELA)

### Funções auxiliares

- `carregar_trechos`, `texto_para_embedding` e `gerar_embeddings`: as mesmas dos outros notebooks do Cap. 5 que as usam. Um trecho por seção; título, seção e conteúdo viram um vetor.
- `PERGUNTAS` e `posicao`: o gabarito (`acervo/perguntas.json`) diz qual trecho responde cada pergunta; `posicao` diz em que lugar ele apareceu.
- `responder`: a geração com citação, usada na Execução 5.

In [ ]:
import re


def carregar_trechos(pasta_acervo: Path) -> list[dict]:
    """Lê os .md do acervo e devolve um trecho por seção "## " (chunking pela estrutura do documento)."""
    trechos = []
    for caminho in sorted(pasta_acervo.rglob("*.md")):
        fonte = caminho.relative_to(pasta_acervo).as_posix()
        doc_id = fonte.removesuffix(".md").replace("/", "_")
        texto = caminho.read_text(encoding="utf-8")
        titulo = texto.splitlines()[0].lstrip("# ").strip()
        # O que vem antes do primeiro "## " é o cabeçalho do documento (título e metadados).
        for numero, secao in enumerate(re.split(r"(?m)^## ", texto)[1:], start=1):
            nome_secao, _, conteudo = secao.partition("\n")
            trechos.append({
                "id": f"{doc_id}_{numero:02d}",
                "doc_id": doc_id,
                "titulo": titulo,
                "secao": nome_secao.strip(),
                "conteudo": conteudo.strip(),
                "fonte": fonte,
            })
    return trechos


def texto_para_embedding(trecho: dict) -> str:
    """O texto que vira vetor: título do documento, seção e conteúdo."""
    return f"{trecho['titulo']} — {trecho['secao']}\n{trecho['conteudo']}"


def gerar_embeddings(textos: list[str], lote: int = 16) -> list[list[float]]:
    vetores = []
    for i in range(0, len(textos), lote):
        resposta = client.embeddings.create(
            model=AZURE_OPENAI_EMBEDDING_DEPLOYMENT,
            input=textos[i:i + lote],
            dimensions=EMBEDDING_DIMENSIONS,
        )
        vetores.extend(item.embedding for item in resposta.data)
    return vetores

In [ ]:
import json

pd.set_option("display.max_colwidth", None)  # tabelas mostram o texto inteiro

PERGUNTAS = {p["id"]: p for p in json.loads((PASTA_ACERVO / "perguntas.json").read_text(encoding="utf-8"))}


def posicao(ids_retornados: list[str], grupo: list[str]) -> int | None:
    """Posição (1, 2, 3...) do primeiro trecho esperado na lista retornada, ou None se ele não apareceu."""
    for i, id_trecho in enumerate(ids_retornados, start=1):
        if id_trecho in grupo:
            return i
    return None


INSTRUCOES_RAG = """
Responda em português, exclusivamente com base nos trechos fornecidos.
Cite a fonte de cada afirmação no formato [id do trecho].
Se os trechos não trouxerem a informação, responda:
'Não há evidência suficiente nos trechos recuperados.'
""".strip()


def responder(pergunta: str, trechos_recuperados: list[dict]) -> str:
    contexto = "\n\n".join(
        f"[{t['id']}] {t['titulo']} — {t['secao']}\n{t['conteudo']}" for t in trechos_recuperados
    )
    resposta = client.responses.create(
        model=AZURE_OPENAI_DEPLOYMENT,
        instructions=INSTRUCOES_RAG,
        input=f"TRECHOS RECUPERADOS:\n{contexto}\n\nPERGUNTA:\n{pergunta}",
        store=False,
    )
    return resposta.output_text

---

# Execução 1 — Preparar os vetores

Os mesmos trechos e o mesmo embedding dos outros notebooks: um trecho por seção `## ` de cada documento, e um vetor de `EMBEDDING_DIMENSIONS` números por trecho, gerado pelo deployment de embedding do Azure OpenAI. Nas seis bases da série, esta etapa é idêntica: só muda onde o vetor vai morar.

In [ ]:
trechos = carregar_trechos(PASTA_ACERVO)
print(f"{len(trechos)} trechos de {len({t['doc_id'] for t in trechos})} documentos")

vetores = gerar_embeddings([texto_para_embedding(t) for t in trechos])
print(f"{len(vetores)} vetores, cada um com {len(vetores[0])} dimensões")

---

# Execução 2 — Criar a estrutura vetorial

No Azure SQL, a "base vetorial" é uma tabela comum: o vetor é mais uma coluna, do tipo `VECTOR(n)`, ao lado do texto. A dimensão faz parte do tipo da coluna: se a dimensão do embedding mudar no `.env` (reinicie o kernel, para o valor novo valer), a tabela precisa ser recriada, e a célula para com um aviso (é o "reindexar tudo" do slide 10). A chave primária é o `id` do trecho: um vetor não pode ser chave.

### Os limites desta base (slide 22)

- **Dimensões, até 1.998:** é o limite do tipo padrão (`float32`). O `text-embedding-3-small` (1.536) cabe. O `text-embedding-3-large` (3.072) só cabe pedindo `dimensions` de até 1.998 na chamada de embedding, ou em meia precisão: `VECTOR(n, float16)` vai até 3.996 e também é GA no Azure SQL Database.
- **Índice, DiskANN:** `CREATE VECTOR INDEX ... WITH (METRIC = 'cosine', TYPE = 'DiskANN')`, consultado com `VECTOR_SEARCH` e `SELECT TOP (N) WITH APPROXIMATE`. É GA no Azure SQL Database, e a versão atual aceita `INSERT`, `UPDATE` e `DELETE` na tabela indexada.
- **Reranking:** não tem; a ordem final é a da distância.
- **Por que não criamos o índice aqui:** ele exige pelo menos 100 linhas com vetor, e o acervo tem 50 trechos. Além disso, a Microsoft recomenda a busca exata quando a consulta compara menos de ~50 mil vetores. Este notebook usa a busca **exata**, com `VECTOR_DISTANCE`.

### O algoritmo do índice (slide 23)

O DiskANN é um grafo otimizado para SSD: recall alto, pouca memória e lida bem com dado que muda. O triângulo continua sendo recall, memória e tempo de indexação. A busca exata deste notebook tem recall perfeito, mas paga por isso: compara a pergunta com todas as linhas a cada consulta, com a CPU do mesmo banco que atende a operação. Em escala, é o custo que, segundo o slide 23, quem promete recall perfeito está escondendo.

In [ ]:
# Slide 22: "até 1.998" é o limite do tipo VECTOR padrão (float32).
if EMBEDDING_DIMENSIONS > 1998:
    raise ValueError(
        f"VECTOR(n) aceita até 1.998 dimensões em float32, e o .env pede {EMBEDDING_DIMENSIONS}. "
        "Peça dimensions de até 1998 ao embedding, ou use VECTOR(n, float16), que vai até 3.996."
    )

cursor = conexao.cursor()
cursor.execute(f"""
IF OBJECT_ID(N'{TABELA}', N'U') IS NULL
    CREATE TABLE {TABELA} (
        id        NVARCHAR(200) NOT NULL PRIMARY KEY,
        doc_id    NVARCHAR(200) NOT NULL,
        titulo    NVARCHAR(400) NOT NULL,
        secao     NVARCHAR(400) NOT NULL,
        conteudo  NVARCHAR(MAX) NOT NULL,
        fonte     NVARCHAR(200) NOT NULL,
        embedding VECTOR({EMBEDDING_DIMENSIONS}) NOT NULL
    );
""")
conexao.commit()

# As colunas como o banco as registrou: sys.columns informa a dimensão e a precisão de cada vetor.
cursor.execute("""
    SELECT name AS coluna, TYPE_NAME(user_type_id) AS tipo, is_nullable AS aceita_nulo,
           vector_dimensions AS dimensoes, vector_base_type_desc AS precisao
    FROM sys.columns
    WHERE object_id = OBJECT_ID(?)
    ORDER BY column_id
""", TABELA)
nomes = [c[0] for c in cursor.description]
# Nas colunas que não são vetor, dimensão e precisão vêm como NULL (None): aparecem em branco.
colunas = [{n: ("" if v is None else v) for n, v in zip(nomes, linha)} for linha in cursor.fetchall()]

print(f"Tabela {TABELA} pronta. Colunas, como o banco as registrou:")
display(pd.DataFrame(colunas))

dimensao_da_tabela = next(c["dimensoes"] for c in colunas if c["coluna"] == "embedding")
if dimensao_da_tabela != EMBEDDING_DIMENSIONS:
    raise RuntimeError(
        f"A tabela já existe com VECTOR({dimensao_da_tabela}), e o .env pede {EMBEDDING_DIMENSIONS} dimensões. "
        "A dimensão faz parte do tipo da coluna: apague a tabela na célula de limpeza e rode o notebook de novo."
    )

---

# Execução 3 — Gravar os trechos

Cada trecho vira uma linha. O `MERGE` usa o `id` do trecho como chave: se a linha existe, atualiza; se não existe, insere. Como o `id` é estável (documento + número da seção), rodar o notebook de novo **atualiza** as mesmas linhas, em vez de duplicar.

O vetor viaja como texto JSON (`[0.0123, -0.0456, ...]`) e vira `VECTOR` dentro do banco, com `CAST`. O JSON de um vetor de 1.536 números passa de 30 mil caracteres, e o driver ODBC envia texto longo como `ntext`, que não converte direto para `VECTOR`. Por isso o parâmetro passa antes por `NVARCHAR(MAX)`.

A gravação é uma transação: as outras conexões só enxergam as linhas depois do `commit()`.

In [ ]:
# Texto JSON → NVARCHAR(MAX) → VECTOR(n). O mesmo trecho de SQL serve para gravar e para buscar.
VETOR_DO_PARAMETRO = f"CAST(CAST(? AS NVARCHAR(MAX)) AS VECTOR({EMBEDDING_DIMENSIONS}))"

SQL_GRAVAR = f"""
MERGE {TABELA} AS destino
USING (SELECT ? AS id, ? AS doc_id, ? AS titulo, ? AS secao, ? AS conteudo, ? AS fonte,
              {VETOR_DO_PARAMETRO} AS embedding) AS origem
    ON destino.id = origem.id
WHEN MATCHED THEN
    UPDATE SET doc_id = origem.doc_id, titulo = origem.titulo, secao = origem.secao,
               conteudo = origem.conteudo, fonte = origem.fonte, embedding = origem.embedding
WHEN NOT MATCHED THEN
    INSERT (id, doc_id, titulo, secao, conteudo, fonte, embedding)
    VALUES (origem.id, origem.doc_id, origem.titulo, origem.secao, origem.conteudo, origem.fonte, origem.embedding);
"""

linhas = [
    (t["id"], t["doc_id"], t["titulo"], t["secao"], t["conteudo"], t["fonte"], json.dumps(v))
    for t, v in zip(trechos, vetores)
]
cursor.executemany(SQL_GRAVAR, linhas)
conexao.commit()

total_na_tabela = cursor.execute(f"SELECT COUNT(*) FROM {TABELA}").fetchval()
print(f"Gravados: {len(linhas)} | linhas na tabela: {total_na_tabela} (trechos no acervo: {len(trechos)})")
if total_na_tabela > len(trechos):
    print("Há linhas a mais: sobras de uma carga anterior. O MERGE atualiza e insere, mas não apaga nada.")

---

# Execução 4 — Buscar por similaridade

A pergunta vira vetor com o mesmo `gerar_embeddings`, e o banco calcula `VECTOR_DISTANCE('cosine', ...)` contra todas as linhas: é a busca **exata** (kNN), sem índice e sem aproximação. A distância de cosseno vai de 0 (vetores idênticos) a 2 (opostos). A função `buscar` devolve **similaridade = 1 − distância**, que é o próprio cosseno: quanto maior, mais próximo, como nas outras bases da série.

Mudar a base não muda a natureza do vetor: o quase empate do SKU 48213-B x C x D do slide 14 continua aqui, agora sem nenhuma aproximação para culpar. Busca híbrida e reranking são decisões de **recuperação** (pasta `tipos-de-rag`); a base é uma decisão de **operação** (slide 21).

### O que esperar e por que pode variar

- Na pergunta em linguagem natural, o trecho "Como pedir o reembolso de uma viagem" tende a vir em 1º.
- No SKU 48213-B, os três SKUs tendem a aparecer juntos no topo, com similaridades que diferem na segunda ou terceira casa decimal: o certo pode ganhar por pouco ou perder para um irmão.
- A busca é exata: com os mesmos vetores, a ordem se repete. Mas cada execução gera de novo os vetores dos trechos e da pergunta, e o modelo, a dimensão e a redação da pergunta mudam o resultado. Num quase empate, uma diferença mínima no vetor basta para inverter os SKUs.

In [ ]:
SQL_BUSCAR = f"""
SELECT TOP (?) id, titulo, secao, conteudo, fonte,
       VECTOR_DISTANCE('cosine', embedding, {VETOR_DO_PARAMETRO}) AS distancia
FROM {TABELA}
ORDER BY distancia;
"""


def buscar(pergunta: str, k: int = 5) -> list[dict]:
    """Busca exata: o banco calcula a distância de cosseno da pergunta para cada linha e devolve as k menores."""
    vetor = gerar_embeddings([pergunta])[0]
    cursor = conexao.cursor()
    cursor.execute(SQL_BUSCAR, k, json.dumps(vetor))
    nomes = [c[0] for c in cursor.description]
    encontrados = []
    for linha in cursor.fetchall():
        r = dict(zip(nomes, linha))
        r["similaridade"] = 1 - r.pop("distancia")  # distância de cosseno (0 a 2) → similaridade (1 a -1)
        encontrados.append(r)
    return encontrados


resultados = {pid: buscar(PERGUNTAS[pid]["pergunta"], k=5) for pid in ["linguagem-natural", "string-exata"]}

for pid, top5 in resultados.items():
    p = PERGUNTAS[pid]
    esperados = p["trechos_esperados"][0]
    lugar = posicao([t["id"] for t in top5], esperados)
    situacao = f"{lugar}º" if lugar else "fora do top 5"
    display(Markdown(f"**{p['pergunta']}** ({p['categoria']}) · posição do trecho certo: **{situacao}**"))
    display(pd.DataFrame([
        {
            "posição": i,
            "trecho": f"{t['titulo']} — {t['secao']}",
            "similaridade": round(t["similaridade"], 4),
            "diferença para o 1º": round(t["similaridade"] - top5[0]["similaridade"], 4),
            "é o trecho certo?": "sim" if t["id"] in esperados else "",
        }
        for i, t in enumerate(top5, start=1)
    ]))

### O que observar

- No SKU, compare a **similaridade** do 1º com a do 2º: é o mesmo quase empate do notebook `tipos-de-rag/01-vetorial-puro`, agora num banco relacional. A régua é outra: lá o `@search.score` é `1 / (1 + distância)`; aqui, `1 − distância`, o próprio cosseno, que deixa maiores as diferenças entre os irmãos, sem que o banco separe melhor os SKUs. Compare a ordem, não os números.
- `TOP (5)` sempre devolve 5 linhas, mesmo quando nenhuma responde de verdade: na busca vetorial pura não existe "não encontrei".
- A consulta é SQL comum: dá para juntar um `WHERE` por documento, área ou permissão na mesma consulta, sobre o mesmo dado que a aplicação de negócio grava.

---

# Execução 5 — RAG completo

O pipeline inteiro, com a recuperação no Azure SQL: os 3 primeiros trechos da pergunta em linguagem natural vão para o modelo, que responde citando o `id` de cada trecho usado. A geração é a mesma das outras bases da série e de `tipos-de-rag`: daqui em diante, trocar a base não muda nada.

In [ ]:
pid = "linguagem-natural"
pergunta = PERGUNTAS[pid]["pergunta"]
top3 = resultados[pid][:3]
lugar = posicao([t["id"] for t in top3], PERGUNTAS[pid]["trechos_esperados"][0])
enviados = ", ".join(f"`{t['id']}`" for t in top3)
situacao = f"sim, em {lugar}º lugar" if lugar else "não"

display(Markdown(f"### {pergunta}"))
display(Markdown(f"**Trechos enviados ao modelo:** {enviados}  \n**O trecho certo está entre eles?** {situacao}"))
display(Markdown(responder(pergunta, top3)))

### O que observar

- Leia a linha **O trecho certo está entre eles?** antes de julgar a resposta: ela separa erro de recuperação de erro de geração (slide 29).

### Leitura arquitetural

- **Slide 22, linha do Azure SQL:** “Já é a base da casa e a dimensão cabe.” As duas condições apareceram aqui: a tabela é mais uma no banco que o time já opera, e o vetor de 1.536 dimensões do `.env.example` cabe no limite de 1.998.
- **Slide 21:** o dado muda em tempo real? Se muda, o vetor vive junto do dado operacional, e o Azure SQL está nessa coluna. O vetor é uma coluna: muda no mesmo `UPDATE` e no mesmo `COMMIT` que o dado do negócio, sem sincronizar com um índice externo.
- **Quem opera, monitora e paga:** o time que já sustenta o banco. Backup, alta disponibilidade, permissões, monitoramento e fatura já existem; não entra componente novo. O custo aparece como CPU do mesmo banco, porque a busca exata varre a tabela a cada pergunta. Acima de ~50 mil vetores por consulta, o índice DiskANN troca exatidão por velocidade.
- **Híbrido aqui:** busca de texto completo (`FREETEXTTABLE`, que exige um índice full-text na tabela) somada à busca vetorial, com a fusão RRF escrita na própria consulta. Não vem pronto, como no AI Search, e não há reranker embutido.

### Fala sugerida

> “A pergunta não é qual banco vetorial é o melhor. É se o dado muda em tempo real.”

Na tabela do slide 22, leia a última coluna, não as do meio (é a nota do próprio slide).

---

# Quadro final — o que muda ao trocar a base?

| Execução | O que vimos | Decisão arquitetural |
|---|---|---|
| 1 | Os mesmos trechos e o mesmo embedding das outras bases | Trocar a base não muda chunking nem embedding |
| 2 | O vetor é uma coluna `VECTOR(n)` numa tabela comum, com a dimensão no tipo | A dimensão cabe? Até 1.998 em `float32` |
| 3 | `MERGE` pelo `id` do trecho, numa transação: rodar de novo não duplica | O vetor muda junto com o dado, no mesmo `COMMIT` |
| 4 | Busca exata com `VECTOR_DISTANCE`, e o quase empate do SKU continua | A base não corrige a recuperação: híbrido e reranking são outra decisão |
| 5 | A resposta com citação, com a recuperação no banco da casa | Sem componente novo: quem já opera o banco opera a recuperação |

## Frase de fechamento

> **Escolher a base é decisão de operação, não de benchmark: cada componente novo precisa de alguém para operar, monitorar e pagar.**

---

# Referências oficiais

- Tipo de dados `vector`: até 1.998 dimensões em float32 (3.996 em float16) e conversão a partir de JSON:  
  https://learn.microsoft.com/sql/t-sql/data-types/vector-data-type

- `VECTOR_DISTANCE`: distância exata, de cosseno de 0 a 2, sem usar índice:  
  https://learn.microsoft.com/sql/t-sql/functions/vector-distance-transact-sql

- Busca vetorial e índices vetoriais no SQL Database Engine: exata x aproximada, DiskANN:  
  https://learn.microsoft.com/sql/sql-server/ai/vectors

- `CREATE VECTOR INDEX`: DiskANN, mínimo de 100 linhas e DML na versão atual:  
  https://learn.microsoft.com/sql/t-sql/statements/create-vector-index-transact-sql

---

# Limpeza (opcional)

A célula abaixo apaga a tabela `dbo.cap05_trechos` quando `APAGAR_RECURSOS = True`. Ela apaga só o que este notebook criou: o banco e o servidor continuam.

Mudou a dimensão do embedding no `.env`? Reinicie o kernel, porque os notebooks leem o `.env` só uma vez. Com a dimensão nova, a Execução 2 para com um aviso, porque a dimensão faz parte do tipo da coluna: apague a tabela aqui e rode o notebook de novo, com `APAGAR_RECURSOS` de volta em `False` (é o "reindexar tudo" do slide 10).

Mudou só o modelo, com a mesma dimensão? Nada falha: reinicie o kernel e rode o notebook inteiro, para regravar os 50 vetores com o modelo novo (slide 10: vetores de modelos diferentes não se comparam).

No fim, a célula fecha a conexão: num banco serverless, uma conexão aberta impede a pausa automática e continua consumindo vCore-segundos (no free tier, do limite mensal gratuito).

In [ ]:
APAGAR_RECURSOS = False  # True apaga a tabela criada por este notebook

if APAGAR_RECURSOS:
    conexao.cursor().execute(f"DROP TABLE IF EXISTS {TABELA}")
    conexao.commit()
    print(f"Tabela {TABELA} apagada.")
else:
    print(f"Nada foi apagado. A tabela {TABELA} continua no banco.")

conexao.close()
print("Conexão fechada.")